# EfficientNet and Galaxy Zoo Dataset Exploration

This notebook was used as an exploratory baseline for looking at Galaxy Zoo and SDSS data: dataset structure, image examples, label columns, and a possible EfficientNet transfer-learning setup.

It is not the final product pipeline. The final playful matcher/card generator uses the curated space-object dataset instead.

Expected Galaxy Zoo structure after downloading from Kaggle:

```text
data/galaxy-zoo-the-galaxy-challenge/
  training_solutions_rev1.csv
  images_training_rev1/
    100008.jpg
    ...
```


In [ ]:
# If dependencies are not installed, uncomment and run:
!pip install -q tensorflow pandas scikit-learn matplotlib pillow kaggle

## Imports and Settings

In [ ]:
from pathlib import Path
import urllib.parse
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.keras.utils.set_random_seed(SEED)

IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

DATA_DIR = Path('data/galaxy-zoo-the-galaxy-challenge')
CSV_PATH = DATA_DIR / 'training_solutions_rev1.csv'
IMG_DIR = DATA_DIR / 'images_training_rev1'

SDSS_DIR = Path('data/sdss_sample')
SDSS_IMG_DIR = SDSS_DIR / 'images'
SDSS_CSV_PATH = SDSS_DIR / 'sdss_galaxies_sample.csv'

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## Downloading Kaggle Data, If Needed

Automatic Kaggle download requires `~/.kaggle/kaggle.json` with an API token. If the data already exists in `DATA_DIR`, skip this cell.

In [ ]:
# Uncomment to download the dataset:
DATA_DIR.mkdir(parents=True, exist_ok=True)
!kaggle competitions download -c galaxy-zoo-the-galaxy-challenge -p {DATA_DIR}
!unzip -q {DATA_DIR}/galaxy-zoo-the-galaxy-challenge.zip -d {DATA_DIR}
!unzip -q {DATA_DIR}/images_training_rev1.zip -d {DATA_DIR}

## Checking and Extracting Galaxy Zoo

This cell searches for `training_solutions_rev1.csv` and the `images_training_rev1` folder. If Kaggle zip files are present, it tries to extract them. If data is missing, it raises a clear instruction message.

In [ ]:
def extract_zip_if_needed(zip_path, target_dir):
    zip_path = Path(zip_path)
    if zip_path.exists():
        print(f'Extracting {zip_path} -> {target_dir}')
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(target_dir)

def find_first(root, pattern):
    matches = sorted(Path(root).glob(pattern))
    return matches[0] if matches else None

DATA_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle may download one combined zip or separate zip files. Support both cases.
extract_zip_if_needed(DATA_DIR / 'galaxy-zoo-the-galaxy-challenge.zip', DATA_DIR)
extract_zip_if_needed(DATA_DIR / 'training_solutions_rev1.zip', DATA_DIR)
extract_zip_if_needed(DATA_DIR / 'images_training_rev1.zip', DATA_DIR)

found_csv = find_first(DATA_DIR, '**/training_solutions_rev1.csv')
found_img_dir = find_first(DATA_DIR, '**/images_training_rev1')

if found_csv is not None:
    CSV_PATH = found_csv
if found_img_dir is not None:
    IMG_DIR = found_img_dir

if not CSV_PATH.exists() or not IMG_DIR.exists():
    raise FileNotFoundError(
        'Galaxy Zoo was not found. Download the Kaggle dataset `galaxy-zoo-the-galaxy-challenge` '
        f'and place the files into {DATA_DIR}. Expected files: `training_solutions_rev1.csv` '
        'and the `images_training_rev1` folder. In Colab: upload kaggle.json, accept the competition rules '
        'on Kaggle, then run the download cell above.'
    )

print('CSV_PATH:', CSV_PATH)
print('IMG_DIR:', IMG_DIR)

## SDSS Sample Export

SDSS can be accessed without Kaggle through the public SkyServer. This block downloads a small mixed SDSS DR18 sample: galaxies, stars, and quasars (`GALAXY`, `STAR`, `QSO`). For each object, it downloads RGB cutout images by `ra` and `dec`. This is not Galaxy Zoo labeling; it is an astronomical catalog with photometry and, where available, redshift.

In [ ]:
SDSS_PER_CLASS = 20
SDSS_CLASSES = ['GALAXY', 'STAR', 'QSO']
SDSS_SEARCH_URL = 'https://skyserver.sdss.org/dr18/SkyServerWS/SearchTools/SqlSearch'

def fetch_sdss_class(spec_class, limit=20):
    sql = f"""
    SELECT TOP {limit}
        p.objID, p.ra, p.dec, p.type AS photo_type,
        p.u, p.g, p.r, p.i, p.z,
        p.petroRad_r, p.petroR50_r, p.petroR90_r,
        s.z AS redshift
    FROM SpecObj AS s
    JOIN PhotoObj AS p ON p.objID = s.bestObjID
    WHERE
        s.class = '{spec_class}'
        AND p.clean = 1
        AND p.r BETWEEN 14 AND 20
    """
    query_url = SDSS_SEARCH_URL + '?' + urllib.parse.urlencode({'cmd': sql, 'format': 'csv'})
    class_df = pd.read_csv(query_url, comment='#')
    class_df['spec_class'] = spec_class
    return class_df

SDSS_DIR.mkdir(parents=True, exist_ok=True)
sdss_parts = []
for spec_class in SDSS_CLASSES:
    print(f'Loading SDSS {spec_class}...')
    sdss_parts.append(fetch_sdss_class(spec_class, SDSS_PER_CLASS))

sdss_df = pd.concat(sdss_parts, ignore_index=True)
sdss_df['photo_type_name'] = sdss_df['photo_type'].map({3: 'photo_galaxy', 6: 'photo_star'}).fillna('photo_other')
sdss_df.to_csv(SDSS_CSV_PATH, index=False)
print(sdss_df.shape)
display(sdss_df['spec_class'].value_counts())
sdss_df.head()

In [ ]:
def sdss_cutout_url(ra, dec, scale=0.2, width=224, height=224):
    params = urllib.parse.urlencode(
        {
            'ra': float(ra),
            'dec': float(dec),
            'scale': scale,
            'width': width,
            'height': height,
            'opt': '',
        }
    )
    return f'https://skyserver.sdss.org/dr18/SkyServerWS/ImgCutout/getjpeg?{params}'

def download_sdss_cutouts(sdss_table, max_images=24):
    SDSS_IMG_DIR.mkdir(parents=True, exist_ok=True)
    image_paths = []

    for row in sdss_table.head(max_images).itertuples(index=False):
        image_path = SDSS_IMG_DIR / f'{int(row.objID)}.jpg'
        if not image_path.exists():
            urllib.request.urlretrieve(sdss_cutout_url(row.ra, row.dec), image_path)
        image_paths.append(str(image_path))

    sdss_table = sdss_table.copy()
    sdss_table.loc[: len(image_paths) - 1, 'image_path'] = image_paths
    return sdss_table

sdss_df = download_sdss_cutouts(sdss_df, max_images=24)
sdss_df.head()

## SDSS Image Examples

In [ ]:
sdss_examples = sdss_df.dropna(subset=['image_path']).head(9)

plt.figure(figsize=(10, 10))
for i, row in enumerate(sdss_examples.itertuples(index=False)):
    plt.subplot(3, 3, i + 1)
    plt.imshow(plt.imread(row.image_path))
    redshift = 'no z' if pd.isna(row.redshift) else f'z={row.redshift:.3f}'
    plt.title(f'{row.spec_class}: r={row.r:.2f}, {redshift}', fontsize=8)
    plt.axis('off')
plt.tight_layout()

## Loading Galaxy Zoo Labels

Galaxy Zoo contains volunteer vote fractions across 37 answer columns. In this exploration notebook, these columns can be treated as multi-output targets; for a simple display, the answer with the highest vote fraction is shown.

In [ ]:
if CSV_PATH.exists() and IMG_DIR.exists():
    df = pd.read_csv(CSV_PATH)
    label_cols = [col for col in df.columns if col != 'GalaxyID']

    df['image_path'] = df['GalaxyID'].astype(str).map(lambda x: str(IMG_DIR / f'{x}.jpg'))
    df = df[df['image_path'].map(lambda p: Path(p).exists())].reset_index(drop=True)

    print(df.shape)
    display(df.head())
else:
    df = pd.DataFrame()
    label_cols = []
    print('Galaxy Zoo was not found. SDSS blocks can still be explored, but Galaxy Zoo is required for training.')

In [ ]:
ANSWER_NAMES = {
    'Class1.1': 'smooth / rounded',
    'Class1.2': 'features or disk',
    'Class1.3': 'star or artifact',
    'Class2.1': 'edge-on disk: yes',
    'Class2.2': 'edge-on disk: no',
    'Class3.1': 'bar: yes',
    'Class3.2': 'bar: no',
    'Class4.1': 'spiral arms: yes',
    'Class4.2': 'spiral arms: no',
    'Class5.1': 'bulge prominence: none',
    'Class5.2': 'bulge prominence: just noticeable',
    'Class5.3': 'bulge prominence: obvious',
    'Class5.4': 'bulge prominence: dominant',
    'Class6.1': 'odd feature: yes',
    'Class6.2': 'odd feature: no',
    'Class7.1': 'rounded: completely',
    'Class7.2': 'rounded: in between',
    'Class7.3': 'rounded: cigar-shaped',
    'Class8.1': 'odd: ring',
    'Class8.2': 'odd: lens or arc',
    'Class8.3': 'odd: disturbed',
    'Class8.4': 'odd: irregular',
    'Class8.5': 'odd: other',
    'Class8.6': 'odd: merger',
    'Class8.7': 'odd: dust lane',
    'Class9.1': 'bulge shape: rounded',
    'Class9.2': 'bulge shape: boxy',
    'Class9.3': 'bulge shape: no bulge',
    'Class10.1': 'spiral winding: tight',
    'Class10.2': 'spiral winding: medium',
    'Class10.3': 'spiral winding: loose',
    'Class11.1': 'spiral arms: 1',
    'Class11.2': 'spiral arms: 2',
    'Class11.3': 'spiral arms: 3',
    'Class11.4': 'spiral arms: 4',
    'Class11.5': 'spiral arms: more than 4',
    'Class11.6': 'spiral arms: cannot tell',
}

def readable_label(col):
    return f'{col} - {ANSWER_NAMES.get(col, col)}'

## Galaxy Zoo Image Examples

In [ ]:
if not df.empty and label_cols:
    galaxy_zoo_examples = df.sample(n=min(9, len(df)), random_state=SEED)

    plt.figure(figsize=(10, 10))
    for i, row in enumerate(galaxy_zoo_examples.itertuples(index=False)):
        plt.subplot(3, 3, i + 1)
        image = plt.imread(row.image_path)
        plt.imshow(image)
        row_values = galaxy_zoo_examples.iloc[i][label_cols].astype(float)
        top_col = row_values.idxmax()
        plt.title(f'{row.GalaxyID}: {readable_label(top_col)}', fontsize=8)
        plt.axis('off')
    plt.tight_layout()
else:
    print('Galaxy Zoo is not loaded. Download the Kaggle dataset first and run the label-loading cell.')

## Dataset Composition

Galaxy Zoo is represented here as images plus 37 probabilistic volunteer-answer columns. SDSS is represented as a mixed object catalog (`GALAXY`, `STAR`, `QSO`) with coordinates, `u/g/r/i/z` photometry, size features, and redshift for objects with spectra.

In [ ]:
galaxy_zoo_image_count = df['image_path'].map(lambda p: Path(p).exists()).sum() if 'image_path' in df else 0
galaxy_zoo_columns = ', '.join(['GalaxyID', 'image_path'] + label_cols[:5]) + ', ...' if label_cols else 'not loaded'

galaxy_zoo_summary = pd.DataFrame(
    {
        'dataset': ['Galaxy Zoo'],
        'rows': [len(df)],
        'image_count': [galaxy_zoo_image_count],
        'label_columns': [len(label_cols)],
        'main_columns': [galaxy_zoo_columns],
    }
)

galaxy_zoo_summary

In [ ]:
if label_cols:
    top_galaxy_zoo_answers = (
        df[label_cols]
        .mean()
        .sort_values(ascending=False)
        .head(12)
        .rename_axis('answer')
        .reset_index(name='mean_vote_fraction')
    )
    top_galaxy_zoo_answers['description'] = top_galaxy_zoo_answers['answer'].map(ANSWER_NAMES)
    display(top_galaxy_zoo_answers)
else:
    print('Galaxy Zoo is not loaded, so answer distribution is unavailable.')

In [ ]:
if 'sdss_df' in globals():
    sdss_summary = pd.DataFrame(
        {
            'dataset': ['SDSS sample'],
            'rows': [len(sdss_df)],
            'image_count': [sdss_df.get('image_path', pd.Series(dtype=str)).notna().sum()],
            'label_columns': [0],
            'main_columns': [', '.join(sdss_df.columns[:10]) + ', ...'],
        }
    )
    display(pd.concat([galaxy_zoo_summary, sdss_summary], ignore_index=True))
else:
    print('Run the SDSS export block above first.')
    display(galaxy_zoo_summary)

In [ ]:
if 'sdss_df' in globals():
    sdss_numeric_cols = ['u', 'g', 'r', 'i', 'z', 'petroRad_r', 'petroR50_r', 'petroR90_r', 'redshift']
    display(sdss_df[sdss_numeric_cols].describe().T)
    display(sdss_df['spec_class'].fillna('no spectrum').value_counts().rename_axis('spec_class').reset_index(name='count'))